# 03 · Analysis and visualisation
**Question:** How much of the October–November PM2.5 rise in Delhi comes from Diwali night, and how much from the season's weather?

Input: `clean/delhi_pm25_hourly.parquet` (11 sites × 3 years × hourly, 15 Sep–30 Nov).

**Plan**
1. Build a city-wide PM2.5 series (average of the 11 sites).
2. Measure the seasonal rise: late-September baseline vs October–November.
3. Ask how much of that rise the weather can explain, using a simple regression of PM2.5 on weather.
4. Measure Diwali night against (a) the nights around it and (b) what the weather alone predicts for that night.
5. Turn the Diwali-night excess into its share of the October–November rise.

Charts are built with Plotly (interactive), shown here and also saved as JSON for the website in `docs/`.

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook_connected"   # charts show in Jupyter/VS Code/Colab and in HTML exports
import statsmodels.formula.api as smf

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CLEAN, META, DOCS = ROOT / "clean", ROOT / "metadata", ROOT / "docs"
FIG_DIR = DOCS / "assets" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

DIWALI = {2023: pd.Timestamp("2023-11-12"), 2024: pd.Timestamp("2024-10-31"), 2025: pd.Timestamp("2025-10-20")}
YEARS = [2023, 2024, 2025]

# ---- Colours (checked with a colour-blind validator) -------------------------
# Years: one blue, light -> dark, plus a different line style each, so they stay apart in greyscale.
YEAR_COLOR = {2023: "#86b6ef", 2024: "#2a78d6", 2025: "#104281"}
YEAR_DASH = {2023: "dot", 2024: "dash", 2025: "solid"}
HIGHLIGHT = "#eb6834"   # the ONE highlight colour: Diwali
CONTEXT = "#b8b7b0"     # neutral grey for background points
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e7e6e2"

def style(fig, title, xtitle, ytitle, height=430):
    fig.update_layout(
        title=dict(text=title, x=0, xanchor="left", font=dict(size=16, color=INK)),
        template="simple_white", height=height, margin=dict(l=60, r=20, t=70, b=55),
        font=dict(family="Inter, Segoe UI, Arial, sans-serif", size=13, color=INK2),
        legend=dict(orientation="h", yanchor="bottom", y=1.0, xanchor="right", x=1, title=None),
        hoverlabel=dict(bgcolor="white", font_size=12), plot_bgcolor="white", paper_bgcolor="white")
    fig.update_xaxes(title=xtitle, showgrid=False, linecolor=GRID, ticks="outside", tickcolor=GRID)
    fig.update_yaxes(title=ytitle, showgrid=True, gridcolor=GRID, linecolor=GRID, zeroline=False)
    return fig

def save(fig, name):
    """Save the chart for the website and show it here."""
    (FIG_DIR / f"{name}.json").write_text(fig.to_json(), encoding="utf-8")
    fig.show()

RESULTS = {}

## 1 · City-wide PM2.5
For each hour we average the 11 sites, but only if at least 8 of them report (so one station's gap cannot swing the city value). Daily means need 18 of 24 hours; "night" means 18:00 to 05:59 and needs 9 of 12 hours.

In [2]:
d = pd.read_parquet(CLEAN / "delhi_pm25_hourly.parquet")
d["station_key"] = d["station_key"].astype(str)
n_sites = d.groupby("ts_ist")["pm25"].count()
city = (d.groupby("ts_ist")["pm25"].mean().where(n_sites >= 8).rename("pm25").to_frame()
          .join(d.drop_duplicates("ts_ist").set_index("ts_ist")[["year", "days_from_diwali", "temp_c", "rh_pct", "rain_mm", "wind_kmh", "blh_m"]]))
print("City hours:", len(city), "| hours without a city value:", city.pm25.isna().sum())

daily = (city.groupby(city.index.normalize())
             .agg(pm25=("pm25", "mean"), n=("pm25", "count"), blh=("blh_m", "mean"), wind=("wind_kmh", "mean"),
                  temp=("temp_c", "mean"), rh=("rh_pct", "mean"), rain=("rain_mm", "sum"), year=("year", "first")))
daily.loc[daily.n < 18, "pm25"] = np.nan
daily["dd"] = [(i - DIWALI[y]).days for i, y in zip(daily.index, daily.year)]
daily["month"] = daily.index.month

city["night_of"] = (city.index - pd.Timedelta(hours=18)).normalize()
nights = (city[(city.index.hour >= 18) | (city.index.hour < 6)].groupby("night_of")
              .agg(pm25=("pm25", "mean"), n=("pm25", "count"), blh=("blh_m", "mean"), wind=("wind_kmh", "mean"),
                   temp=("temp_c", "mean"), rh=("rh_pct", "mean"), year=("year", "first")))
nights.loc[nights.n < 9, "pm25"] = np.nan
nights["dd"] = [(i - DIWALI[y]).days for i, y in zip(nights.index, nights.year)]
nights = nights[nights.index.month * 100 + nights.index.day <= 1129]   # the 30 Nov night runs past the window
print("Days:", len(daily), "| nights:", len(nights))

City hours: 5544 | hours without a city value: 0
Days: 231 | nights: 231


## 2 · Summary statistics and the seasonal rise

In [3]:
def period(m):
    return np.where(m == 9, "15-30 Sep", "Oct-Nov")
city["period"] = period(city.index.month)
summary = (city.groupby(["year", "period"])["pm25"].agg(["mean", "median", "max", "count"]).round(0).unstack())
rise = (city[city.period == "Oct-Nov"].groupby("year").pm25.mean() - city[city.period == "15-30 Sep"].groupby("year").pm25.mean()).round(1)
summary[("rise", "Oct-Nov minus Sep")] = rise
summary

mean            median               max             count  \
period 15-30 Sep Oct-Nov 15-30 Sep Oct-Nov 15-30 Sep Oct-Nov 15-30 Sep   
year                                                                     
2023        47.0   180.0      48.0   160.0      99.0   679.0       384   
2024        52.0   184.0      47.0   167.0     139.0   857.0       384   
2025        47.0   174.0      45.0   151.0     113.0  1331.0       384   

                            rise  
period Oct-Nov Oct-Nov minus Sep  
year                              
2023      1464             132.9  
2024      1464             131.7  
2025      1464             126.3

In [4]:
RESULTS["sep_mean"] = city[city.period == "15-30 Sep"].groupby("year").pm25.mean().round(1).to_dict()
RESULTS["octnov_mean"] = city[city.period == "Oct-Nov"].groupby("year").pm25.mean().round(1).to_dict()
RESULTS["rise"] = rise.to_dict()
NAAQS_24H = 60   # India's 24-hour standard for PM2.5 (µg/m³)
over = daily[daily.month >= 10].groupby("year").apply(lambda x: (x.pm25 > NAAQS_24H).sum() / x.pm25.notna().sum() * 100).round(0)
RESULTS["pct_days_over_naaqs_octnov"] = over.to_dict()
print("Share of Oct-Nov days above India's 24-h standard (60):", over.to_dict())

# By station: Oct-Nov mean
st = d[d.ts_ist.dt.month >= 10].groupby(["station_key", "year"]).pm25.mean().unstack().round(0)
st["mean"] = st.mean(axis=1).round(0)
st.sort_values("mean", ascending=False)

Share of Oct-Nov days above India's 24-h standard (60): {2023: 95.0, 2024: 93.0, 2025: 89.0}


/sessions/rcw-01a8zy71zkermwkg9fl4twy5/tmp/ipykernel_11/1795557951.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  over = daily[daily.month >= 10].groupby("year").apply(lambda x: (x.pm25 > NAAQS_24H).sum() / x.pm25.notna().sum() * 100).round(0)


year,2023,2024,2025,mean
station_key,,,,
jahangirpuri,216.0,222.0,210.0,216.0
anand_vihar,209.0,232.0,198.0,213.0
punjabi_bagh,207.0,206.0,189.0,201.0
rk_puram,195.0,190.0,186.0,190.0
narela,183.0,192.0,188.0,188.0
dwarka_sec8,188.0,189.0,173.0,183.0
okhla_ph2,179.0,175.0,163.0,172.0
ito,168.0,163.0,176.0,169.0
najafgarh,158.0,164.0,141.0,154.0


### Chart 1 (univariate): how the distribution of hourly PM2.5 shifts from September to November

In [5]:
city["month_name"] = city.index.month.map({9: "September (15-30)", 10: "October", 11: "November"})
fig1 = go.Figure()
for y in YEARS:
    sub = city[city.year == y]
    fig1.add_trace(go.Box(x=sub.month_name, y=sub.pm25, name=str(y), marker_color=YEAR_COLOR[y], line=dict(width=1.5),
                          boxpoints=False, hovertemplate="%{x}<br>%{y:.0f} µg/m³<extra>" + str(y) + "</extra>"))
fig1.update_layout(boxmode="group")
fig1.add_hline(y=NAAQS_24H, line=dict(color=INK2, width=1, dash="dot"),
               annotation_text="India 24-h standard (60)", annotation_position="top left", annotation_font_color=INK2)
style(fig1, "PM2.5 roughly quadruples between late September and November, every year",
      "Month", "Hourly city-wide PM2.5 (µg/m³)")
save(fig1, "fig1_monthly_distribution")

### Chart 2: the whole season, day by day

In [6]:
fig2 = go.Figure()
for y in YEARS:
    sub = daily[daily.year == y]
    x = pd.to_datetime("2000-" + sub.index.strftime("%m-%d"))   # common calendar so years overlay
    fig2.add_trace(go.Scatter(x=x, y=sub.pm25, name=str(y), mode="lines", line=dict(color=YEAR_COLOR[y], width=2, dash=YEAR_DASH[y]),
                              customdata=sub.index.strftime("%d %b %Y"),
                              hovertemplate="%{customdata}<br>%{y:.0f} µg/m³<extra></extra>"))
    dw = DIWALI[y]
    fig2.add_trace(go.Scatter(x=[pd.Timestamp(2000, dw.month, dw.day)], y=[daily.loc[dw, "pm25"]], mode="markers+text",
                              marker=dict(color=HIGHLIGHT, size=11, line=dict(color="white", width=2)),
                              text=[f"Diwali {y}"], textposition="top center", textfont=dict(color=INK, size=11),
                              showlegend=False, hovertemplate=f"Diwali {y}<br>%{{y:.0f}} µg/m³ (daily mean)<extra></extra>"))
fig2.add_hline(y=NAAQS_24H, line=dict(color=INK2, width=1, dash="dot"), annotation_text="India 24-h standard (60)",
               annotation_position="bottom right", annotation_font_color=INK2)
style(fig2, "The rise starts in mid-October and peaks in November, whenever Diwali falls",
      "Date (same calendar for all three years)", "Daily city-wide PM2.5 (µg/m³)")
fig2.update_xaxes(tickformat="%d %b")
save(fig2, "fig2_season_daily")

## 3 · How much of the seasonal rise does the weather explain?
As the season moves into winter, the air near the ground gets colder and stiller, and the **boundary-layer height** (how deep the mixing layer is) drops, so smoke has less room to spread. First we simply look at how strongly each weather variable moves with daily PM2.5.

In [7]:
rho_d = daily[daily.pm25.notna() & ~daily.dd.isin([0, 1])][["pm25", "blh", "wind", "temp", "rh", "rain"]].corr(method="spearman")["pm25"].drop("pm25")
RESULTS["spearman_daily"] = rho_d.round(2).to_dict()
rho_d.round(2).to_frame("Spearman correlation with daily PM2.5")

,Spearman correlation with daily PM2.5
blh,-0.21
wind,-0.36
temp,-0.74
rh,-0.30
rain,-0.46


### Chart 3 (bivariate): daily PM2.5 against three weather variables

In [8]:
from plotly.subplots import make_subplots
panels = [("blh", "Mixing-layer height (m, log)", True), ("wind", "Wind speed (km/h)", False), ("temp", "Air temperature (°C)", False)]
fig3 = make_subplots(rows=1, cols=3, shared_yaxes=True, horizontal_spacing=0.04,
                     subplot_titles=[f"{t.split(' (')[0]}: ρ = {rho_d[c]:.2f}" for c, t, _ in panels])
is_dw_day = daily.dd == 0
for i, (c, t, logx) in enumerate(panels, start=1):
    fig3.add_trace(go.Scatter(x=daily.loc[~is_dw_day, c], y=daily.loc[~is_dw_day, "pm25"], mode="markers", name="Other days",
                              marker=dict(color=CONTEXT, size=7, opacity=0.8, line=dict(color="white", width=1)), showlegend=(i == 1),
                              customdata=daily.index[~is_dw_day].strftime("%d %b %Y"),
                              hovertemplate="%{customdata}<br>%{x:.1f}<br>PM2.5 %{y:.0f} µg/m³<extra></extra>"), row=1, col=i)
    fig3.add_trace(go.Scatter(x=daily.loc[is_dw_day, c], y=daily.loc[is_dw_day, "pm25"], mode="markers", name="Diwali day",
                              marker=dict(color=HIGHLIGHT, size=11, line=dict(color="white", width=2)), showlegend=(i == 1),
                              customdata=daily.index[is_dw_day].strftime("%d %b %Y"),
                              hovertemplate="Diwali %{customdata}<br>%{x:.1f}<br>PM2.5 %{y:.0f} µg/m³<extra></extra>"), row=1, col=i)
    fig3.update_xaxes(title_text=t, type="log" if logx else "linear", row=1, col=i)
style(fig3, "Daily PM2.5 tracks temperature closely; mixing height and wind only loosely", None, None, height=470)
fig3.update_yaxes(title_text="Daily city PM2.5 (µg/m³)", row=1, col=1)
for i, (c, t, logx) in enumerate(panels, start=1):
    fig3.update_xaxes(title_text=t, type="log" if logx else "linear", row=1, col=i)
fig3.update_layout(legend=dict(orientation="h", y=-0.28, yanchor="top", x=0, xanchor="left"), margin=dict(b=110))
for a in fig3.layout.annotations:
    a.font = dict(size=13, color=INK)
save(fig3, "fig3_weather_scatter")

**What the correlations say.** Temperature moves closely with PM2.5 (ρ ≈ −0.7), while mixing height and wind move only loosely with it (ρ between −0.2 and −0.35). But temperature is also a clock: it falls steadily from September to November, exactly when crop-residue burning in Punjab and Haryana and winter heating ramp up. We have no data on those. So "temperature" partly stands in for "the time of year", not only for "the weather".

That is why we fit **two** models of daily PM2.5 (using every day except Diwali and the day after) and treat the truth as lying between them:

| Model | Formula | What it counts as "weather" |
|---|---|---|
| **Ventilation only** (lower estimate) | `log(PM2.5) = a + b·log(mixing height) + c·wind` | Only how well the air can carry pollution away |
| **Full weather** (upper estimate) | `… + d·temperature + e·humidity` | Also the cooling of the season, which overlaps with stubble burning |

We use the log of PM2.5 because weather effects are multiplicative (a lid twice as low roughly doubles the concentration at any level). Then we ask each model: *given only the weather, how much higher should October–November be than late September?*

In [9]:
fit_days = daily[~daily.dd.isin([0, 1]) & daily.pm25.notna()].copy()
MODELS = {"ventilation": "np.log(pm25) ~ np.log(blh) + wind",
          "full": "np.log(pm25) ~ np.log(blh) + wind + temp + rh"}
fits, loyo = {}, []
for name, f in MODELS.items():
    m = smf.ols(f, data=fit_days).fit()
    fits[name] = m
    smear = np.exp(m.resid).mean()          # converts a log-scale prediction back to µg/m³ without bias
    daily[f"pred_{name}"] = np.exp(m.predict(daily)) * smear
    for y in YEARS:                          # out-of-sample check: train on two years, predict the third
        mm = smf.ols(f, data=fit_days[fit_days.year != y]).fit()
        t = fit_days[fit_days.year == y]; lp = np.log(t.pm25)
        loyo.append({"model": name, "left_out_year": y, "R2_unseen_year": 1 - ((lp - mm.predict(t)) ** 2).sum() / ((lp - lp.mean()) ** 2).sum()})
print(fits["full"].summary().tables[1])
fit_table = pd.DataFrame({n: {"R2 (all days)": m.rsquared, "days": int(m.nobs)} for n, m in fits.items()}).T
loyo = pd.DataFrame(loyo).pivot(index="model", columns="left_out_year", values="R2_unseen_year")
RESULTS["model_r2"] = {n: round(m.rsquared, 2) for n, m in fits.items()}
RESULTS["model_r2_unseen_year"] = loyo.round(2).to_dict(orient="index")
pd.concat([fit_table, loyo.add_prefix("R2 unseen ")], axis=1).round(2)

                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept       8.5742      0.934      9.181      0.000       6.734      10.415
np.log(blh)     0.1464      0.155      0.943      0.346      -0.159       0.452
wind           -0.0672      0.014     -4.975      0.000      -0.094      -0.041
temp           -0.1258      0.009    -13.290      0.000      -0.144      -0.107
rh             -0.0180      0.003     -5.745      0.000      -0.024      -0.012


,R2 (all days),days,R2 unseen 2023,R2 unseen 2024,R2 unseen 2025
ventilation,0.20,225.0,0.15,0.18,-0.23
full,0.63,225.0,0.63,0.48,0.44


In [10]:
dec = []
for y in YEARS:
    x = daily[daily.year == y]
    sep, on = x.month == 9, x.month >= 10
    row = {"year": y, "observed_sep": x[sep].pm25.mean(), "observed_octnov": x[on].pm25.mean()}
    row["observed_rise"] = row["observed_octnov"] - row["observed_sep"]
    for n in MODELS:
        row[f"rise_{n}"] = x[on][f"pred_{n}"].mean() - x[sep][f"pred_{n}"].mean()
        row[f"share_{n}_%"] = row[f"rise_{n}"] / row["observed_rise"] * 100
    dec.append(row)
dec = pd.DataFrame(dec).set_index("year")
RESULTS["weather_rise"] = dec[["rise_ventilation", "rise_full"]].round(1).to_dict(orient="index")
RESULTS["weather_share_pct"] = dec[["share_ventilation_%", "share_full_%"]].round(0).to_dict(orient="index")
dec.round(1)

,observed_sep,observed_octnov,observed_rise,rise_ventilation,share_ventilation_%,rise_full,share_full_%
year,,,,,,,
2023,47.0,179.9,132.9,4.6,3.5,96.7,72.8
2024,51.9,183.6,131.7,22.9,17.4,95.0,72.1
2025,47.5,173.8,126.3,52.2,41.3,154.3,122.2


**Reading the result.** Ventilation alone (mixing height and wind) predicts only a small part of the rise, and does poorly on a year it has not seen. Adding temperature and humidity makes the model predict most or even all of the rise (more than 100% in 2025, which means it over-shoots). The honest summary: **the seasonal rise is driven by the season, and our weather data cannot cleanly split "stiller, colder air" from "more smoke arriving at this time of year"**. What we *can* pin down is the part that is clearly not the season: Diwali night.

## 4 · Diwali night
"Diwali night" = 18:00 on Diwali day to 05:59 the next morning. We compare it with two baselines:
- **(a) Neighbouring nights:** the average of the nights 2–7 days before and 2–7 days after (skipping the night before and after, which can carry early or leftover firework smoke).
- **(b) Weather-expected:** what a model of night-time PM2.5 on night-time weather predicts for that exact night. This accounts for Diwali falling on a still or a breezy night.

In [11]:
fit_n = nights[~nights.dd.isin([0, 1]) & nights.pm25.notna()]
m_night = smf.ols("np.log(pm25) ~ np.log(blh) + wind + temp + rh", data=fit_n).fit()
smear_n = np.exp(m_night.resid).mean()
nights["pred_weather"] = np.exp(m_night.predict(nights)) * smear_n
print(f"Night model R² = {m_night.rsquared:.2f} on {int(m_night.nobs)} nights")

rows = []
for y in YEARS:
    x = nights[nights.year == y]
    obs = x.loc[x.dd == 0, "pm25"].iloc[0]
    neigh = x[x.dd.abs().between(2, 7)].pm25.mean()
    wexp = x.loc[x.dd == 0, "pred_weather"].iloc[0]
    rows.append({"year": y, "diwali_night": obs, "neighbour_nights": neigh, "excess_vs_neighbours": obs - neigh,
                 "weather_expected": wexp, "excess_vs_weather": obs - wexp, "times_weather_expected": obs / wexp,
                 "blh_m": x.loc[x.dd == 0, "blh"].iloc[0], "wind_kmh": x.loc[x.dd == 0, "wind"].iloc[0]})
dw = pd.DataFrame(rows).set_index("year")
RESULTS["diwali"] = dw.round(1).to_dict(orient="index")
dw.round(1)

Night model R² = 0.65 on 222 nights


,diwali_night,neighbour_nights,excess_vs_neighbours,weather_expected,excess_vs_weather,times_weather_expected,blh_m,wind_kmh
year,,,,,,,,
2023,463.1,271.2,191.9,258.3,204.8,1.8,49.6,6.5
2024,382.1,208.6,173.5,110.3,271.8,3.5,114.2,10.1
2025,785.0,141.8,643.2,93.5,691.5,8.4,170.8,3.7


### From "Diwali night is X µg/m³ higher" to "Diwali night's share of the season"
Diwali night is 12 hours long. October–November is 61 days = 1,464 hours. If Diwali night adds an excess of *E* µg/m³ for 12 hours, it raises the October–November average by *E × 12 / 1,464*. Dividing that by the seasonal rise gives Diwali night's share of the rise. We use the more conservative of the two excess estimates (the smaller one) as our main figure and show both.

In [12]:
HOURS_ON = 61 * 24
share = []
for y in YEARS:
    for basis in ["excess_vs_neighbours", "excess_vs_weather"]:
        e = dw.loc[y, basis]
        add = e * 12 / HOURS_ON
        share.append({"year": y, "basis": basis.replace("excess_vs_", "vs "), "excess_ugm3": e,
                      "adds_to_octnov_mean": add, "share_of_rise_%": add / dec.loc[y, "observed_rise"] * 100})
share = pd.DataFrame(share)
main = share.loc[share.groupby("year").excess_ugm3.idxmin()].set_index("year")
RESULTS["diwali_share_of_rise_pct"] = main["share_of_rise_%"].round(1).to_dict()
RESULTS["diwali_share_range_pct"] = share.groupby("year")["share_of_rise_%"].agg(["min", "max"]).round(1).to_dict(orient="index")
share.round(1)

,year,basis,excess_ugm3,adds_to_octnov_mean,share_of_rise_%
0,2023,vs neighbours,191.9,1.6,1.2
1,2023,vs weather,204.8,1.7,1.3
2,2024,vs neighbours,173.5,1.4,1.1
3,2024,vs weather,271.8,2.2,1.7
4,2025,vs neighbours,643.2,5.3,4.2
5,2025,vs weather,691.5,5.7,4.5


### Chart 4: Diwali night, hour by hour

In [13]:
fig4 = go.Figure()
for y in YEARS:
    start = DIWALI[y] - pd.Timedelta(days=3)
    sub = city.loc[start: DIWALI[y] + pd.Timedelta(days=4)]
    hrs = (sub.index - DIWALI[y]) / pd.Timedelta(hours=1)
    fig4.add_trace(go.Scatter(x=hrs, y=sub.pm25, name=str(y), mode="lines", line=dict(color=YEAR_COLOR[y], width=2, dash=YEAR_DASH[y]),
                              customdata=sub.index.strftime("%d %b %Y %H:%M"),
                              hovertemplate="%{customdata}<br>%{y:.0f} µg/m³<extra></extra>"))
fig4.add_vrect(x0=18, x1=30, fillcolor=HIGHLIGHT, opacity=0.12, line_width=0,
               annotation_text="Diwali night (18:00-06:00)", annotation_position="top left", annotation_font_color=INK)
style(fig4, "On Diwali night the city's PM2.5 jumps within a few hours, then takes about a day to fall back",
      "Hours from midnight at the start of Diwali day", "Hourly city-wide PM2.5 (µg/m³)")
fig4.update_xaxes(tickvals=list(range(-72, 97, 24)), ticktext=["-3 d", "-2 d", "-1 d", "Diwali day", "+1 d", "+2 d", "+3 d", "+4 d"])
save(fig4, "fig4_diwali_hourly")

### Chart 5: Diwali night against what the weather predicts

In [14]:
fig5 = go.Figure()
xs = [str(y) for y in YEARS]
fig5.add_trace(go.Bar(x=xs, y=dw.neighbour_nights, name="Neighbouring nights (avg)", marker_color=CONTEXT,
                      hovertemplate="%{x}: %{y:.0f} µg/m³<extra>Neighbouring nights</extra>"))
fig5.add_trace(go.Bar(x=xs, y=dw.weather_expected, name="Expected from that night's weather", marker_color=YEAR_COLOR[2024],
                      hovertemplate="%{x}: %{y:.0f} µg/m³<extra>Weather-expected</extra>"))
fig5.add_trace(go.Bar(x=xs, y=dw.diwali_night, name="Actual Diwali night", marker_color=HIGHLIGHT,
                      text=[f"{v:.0f}" for v in dw.diwali_night], textposition="outside", textfont=dict(color=INK),
                      hovertemplate="%{x}: %{y:.0f} µg/m³<extra>Diwali night</extra>"))
fig5.update_traces(marker_line_width=0)
fig5.update_layout(barmode="group", bargap=0.3, bargroupgap=0.08)
style(fig5, "Diwali night was roughly 2 to 8 times what that night's weather would predict", "Year", "Night-time city PM2.5 (µg/m³)")
save(fig5, "fig5_diwali_vs_expected")

### Chart 6: the answer to the question
For each year: the October–November rise over late September, the part the weather model predicts, and the part Diwali night adds to the season average.

In [15]:
fig6 = go.Figure()
fig6.add_trace(go.Bar(y=xs, x=dec.observed_rise, name="Observed rise (Oct-Nov minus late Sep)", orientation="h", marker_color=CONTEXT,
                      hovertemplate="%{y}: +%{x:.0f} µg/m³<extra>Observed rise</extra>"))
fig6.add_trace(go.Bar(y=xs, x=dec.rise_ventilation, name="Predicted by ventilation only (mixing height, wind)", orientation="h",
                      marker_color=YEAR_COLOR[2023], hovertemplate="%{y}: +%{x:.0f} µg/m³<extra>Ventilation-only model</extra>"))
fig6.add_trace(go.Bar(y=xs, x=dec.rise_full, name="Predicted by full weather (+ temperature, humidity)", orientation="h",
                      marker_color=YEAR_COLOR[2025], hovertemplate="%{y}: +%{x:.0f} µg/m³<extra>Full-weather model</extra>"))
fig6.add_trace(go.Bar(y=xs, x=main.adds_to_octnov_mean, name="Added by Diwali night", orientation="h", marker_color=HIGHLIGHT,
                      text=[f"+{v:.1f}" for v in main.adds_to_octnov_mean], textposition="outside", textfont=dict(color=INK),
                      hovertemplate="%{y}: +%{x:.1f} µg/m³<extra>Diwali night</extra>"))
fig6.update_layout(barmode="group", bargap=0.2, bargroupgap=0.05)
style(fig6, "The season, not Diwali night, drives the October-November rise",
      "Increase in Oct-Nov average PM2.5 over late September (µg/m³)", "", height=520)
fig6.update_yaxes(autorange="reversed", showgrid=False)
fig6.update_xaxes(showgrid=True, gridcolor=GRID)
fig6.update_layout(legend=dict(orientation="v", y=-0.2, yanchor="top", x=0, xanchor="left"), margin=dict(b=150))
save(fig6, "fig6_decomposition")

### Chart 7: which neighbourhoods were hit hardest on Diwali night?

In [16]:
d["night_of"] = (d.ts_ist - pd.Timedelta(hours=18)).dt.normalize()
dn = d[(d.ts_ist.dt.hour >= 18) | (d.ts_ist.dt.hour < 6)]
sn = dn.groupby(["station_key", "night_of"]).pm25.agg(["mean", "count"]).reset_index()
sn = sn[sn["count"] >= 9]
sn["year"] = sn.night_of.dt.year
sn["dd"] = [(n - DIWALI[y]).days for n, y in zip(sn.night_of, sn.year)]
st_ex = []
for (k, y), g in sn.groupby(["station_key", "year"]):
    if (g.dd == 0).any():
        st_ex.append({"station": k, "year": y, "excess": g.loc[g.dd == 0, "mean"].iloc[0] - g[g.dd.abs().between(2, 7)]["mean"].mean()})
st_ex = pd.DataFrame(st_ex)
NAMES = {"anand_vihar": "Anand Vihar", "rk_puram": "R K Puram", "okhla_ph2": "Okhla Ph-2", "punjabi_bagh": "Punjabi Bagh",
         "najafgarh": "Najafgarh", "ito": "ITO", "pusa": "Pusa", "lodhi_road_imd": "Lodhi Road", "jahangirpuri": "Jahangirpuri",
         "narela": "Narela", "dwarka_sec8": "Dwarka Sec-8"}
st_ex["name"] = st_ex.station.map(NAMES)
order = st_ex.groupby("name").excess.mean().sort_values().index.tolist()
fig7 = go.Figure()
for y in YEARS:
    s = st_ex[st_ex.year == y].set_index("name").reindex(order)
    fig7.add_trace(go.Scatter(x=s.excess, y=order, mode="markers", name=str(y),
                              marker=dict(color=YEAR_COLOR[y], size=11, symbol={2023: "circle-open", 2024: "diamond", 2025: "circle"}[y],
                                          line=dict(width=2, color=YEAR_COLOR[y])),
                              hovertemplate="%{y}<br>+%{x:.0f} µg/m³<extra>" + str(y) + "</extra>"))
style(fig7, "2025 was the worst Diwali at almost every station; the outer-edge stations Narela and Najafgarh barely rose in 2023-24",
      "Diwali-night PM2.5 minus neighbouring nights (µg/m³)", "", height=470)
fig7.update_xaxes(showgrid=True, gridcolor=GRID); fig7.update_yaxes(showgrid=False)
save(fig7, "fig7_station_excess")
RESULTS["station_excess_range"] = st_ex.groupby("year").excess.agg(["min", "max"]).round(0).to_dict(orient="index")
st_ex.pivot(index="name", columns="year", values="excess").round(0).loc[order[::-1]]

year,2023,2024,2025
name,,,
R K Puram,NaN,178.0,961.0
Okhla Ph-2,NaN,341.0,784.0
Pusa,288.0,265.0,903.0
ITO,421.0,273.0,673.0
Dwarka Sec-8,136.0,163.0,1003.0
Lodhi Road,213.0,291.0,598.0
Anand Vihar,264.0,216.0,511.0
Punjabi Bagh,76.0,117.0,480.0
Jahangirpuri,244.0,62.0,352.0


## 5 · Data-quality charts
### Chart 8: before and after cleaning (one station where cleaning mattered)

In [17]:
ito = d[(d.station_key == "ito") & (d.year == 2025) & (d.ts_ist.between("2025-11-18", "2025-11-30 23:00"))]
fig8 = go.Figure()
fig8.add_trace(go.Scatter(x=ito.ts_ist, y=ito.pm25_raw, name="Before cleaning (raw)", mode="lines",
                          line=dict(color=CONTEXT, width=2), hovertemplate="%{x|%d %b %H:%M}<br>raw %{y:.0f}<extra></extra>"))
fig8.add_trace(go.Scatter(x=ito.ts_ist, y=ito.pm25, name="After cleaning", mode="lines",
                          line=dict(color=YEAR_COLOR[2025], width=2), hovertemplate="%{x|%d %b %H:%M}<br>clean %{y:.0f}<extra></extra>"))
bad = ito[ito.pm25_flag.isin(["impossible_value", "stuck", "isolated_spike"])]
fig8.add_trace(go.Scatter(x=bad.ts_ist, y=bad.pm25_raw, name="Removed (error)", mode="markers",
                          marker=dict(color=HIGHLIGHT, size=8, symbol="x"),
                          customdata=bad.pm25_flag.astype(str), hovertemplate="%{x|%d %b %H:%M}<br>%{y:.0f} - %{customdata}<extra></extra>"))
fig8.add_hline(y=0, line=dict(color=INK2, width=1))
style(fig8, "Cleaning removed impossible readings: PM2.5 of exactly zero at ITO, late November 2025",
      "Date and hour (IST)", "PM2.5 (µg/m³)")
save(fig8, "fig8_before_after")
print("Removed points in this window:", len(bad), bad.pm25_flag.value_counts().to_dict())

Removed points in this window: 4 {'impossible_value': 4, 'citywide_high_kept': 0, 'isolated_spike': 0, 'ok': 0, 'stuck': 0}


In [18]:
before = d.assign(m=d.pm25_raw.isna()).groupby("station_key").m.mean() * 100
after = d.assign(m=d.pm25.isna()).groupby("station_key").m.mean() * 100
qa = pd.DataFrame({"missing before %": before, "missing after %": after}).round(1)
qa.index = qa.index.map(NAMES)
removed = d[d.pm25_flag.isin(["impossible_value", "stuck", "isolated_spike"])].groupby("station_key").size()
qa["errors removed"] = removed.rename(index=NAMES).reindex(qa.index).fillna(0).astype(int)
RESULTS["cleaning_counts"] = d.pm25_flag.value_counts().to_dict()
RESULTS["source_counts"] = d.pm25_source.value_counts().to_dict()
qa.sort_values("missing before %", ascending=False)

,missing before %,missing after %,errors removed
station_key,,,
Anand Vihar,8.3,5.7,13
Lodhi Road,5.4,5.8,38
R K Puram,4.4,3.4,4
Punjabi Bagh,3.6,3.1,14
Najafgarh,3.4,2.7,7
ITO,2.8,2.5,30
Okhla Ph-2,2.1,1.4,0
Jahangirpuri,1.5,1.0,0
Dwarka Sec-8,1.0,0.7,8


### Chart 9: CPCB vs OpenAQ for the same stations (2025)

In [19]:
match = pd.read_csv(META / "station_match_cpcb_openaq.csv")
fig9 = go.Figure()
mm = match.copy(); mm["name"] = mm.station_key.map(lambda k: NAMES.get(k, k.replace("pusa_dpcc", "Pusa (DPCC)").replace("pusa_iitm", "Pusa (IITM)")))
mm = mm.sort_values("share_within_0.5")
fig9.add_trace(go.Bar(x=mm["share_within_0.5"] * 100, y=mm["name"], orientation="h", marker_color=YEAR_COLOR[2024],
                      text=[f"{v:.0f}%" for v in mm["share_within_0.5"] * 100], textposition="outside", textfont=dict(color=INK),
                      customdata=np.stack([mm.mean_abs_diff, mm.hours_compared], axis=1),
                      hovertemplate="%{y}<br>%{x:.1f}% of hours identical (±0.5)<br>mean difference %{customdata[0]:.2f} µg/m³<br>%{customdata[1]} hours compared<extra></extra>"))
fig9.add_vline(x=80, line=dict(color=HIGHLIGHT, width=2, dash="dash"), annotation_text="match rule: 80%", annotation_position="bottom right", annotation_font_color=INK)
style(fig9, "OpenAQ is a copy of CPCB data: 11 of 12 stations agree hour-for-hour in at least 80% of hours", "Hours where the two sources agree within 0.5 µg/m³ (%)", "", height=470)
fig9.update_xaxes(range=[0, 112], showgrid=True, gridcolor=GRID); fig9.update_yaxes(showgrid=False)
save(fig9, "fig9_source_agreement")

## 6 · Save the numbers the website and report use

In [20]:
def clean_keys(o):
    if isinstance(o, dict):
        return {str(k): clean_keys(v) for k, v in o.items()}
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    return o
(DOCS / "assets" / "results.json").write_text(json.dumps(clean_keys(RESULTS), indent=1), encoding="utf-8")
print(json.dumps(clean_keys(RESULTS), indent=1))

{
 "sep_mean": {
  "2023": 47.0,
  "2024": 51.9,
  "2025": 47.5
 },
 "octnov_mean": {
  "2023": 179.9,
  "2024": 183.6,
  "2025": 173.8
 },
 "rise": {
  "2023": 132.9,
  "2024": 131.7,
  "2025": 126.3
 },
 "pct_days_over_naaqs_octnov": {
  "2023": 95.0,
  "2024": 93.0,
  "2025": 89.0
 },
 "spearman_daily": {
  "blh": -0.21,
  "wind": -0.36,
  "temp": -0.74,
  "rh": -0.3,
  "rain": -0.46
 },
 "model_r2": {
  "ventilation": 0.2,
  "full": 0.63
 },
 "model_r2_unseen_year": {
  "full": {
   "2023": 0.63,
   "2024": 0.48,
   "2025": 0.44
  },
  "ventilation": {
   "2023": 0.15,
   "2024": 0.18,
   "2025": -0.23
  }
 },
 "weather_rise": {
  "2023": {
   "rise_ventilation": 4.6,
   "rise_full": 96.7
  },
  "2024": {
   "rise_ventilation": 22.9,
   "rise_full": 95.0
  },
  "2025": {
   "rise_ventilation": 52.2,
   "rise_full": 154.3
  }
 },
 "weather_share_pct": {
  "2023": {
   "share_ventilation_%": 3.0,
   "share_full_%": 73.0
  },
  "2024": {
   "share_ventilation_%": 17.0,
   "share_full_

## 7 · Bundle the charts for the website
The website in `docs/` must also open offline (double-clicking `index.html`), where browsers block loading separate JSON files. So we pack all figures into one JavaScript file, and save the matching Plotly library next to it.

In [21]:
from plotly.offline import get_plotlyjs
figs = {p.stem: json.loads(p.read_text(encoding="utf-8")) for p in sorted(FIG_DIR.glob("fig*.json"))}
(DOCS / "assets" / "figures.js").write_text("window.FIGS = " + json.dumps(figs) + ";", encoding="utf-8")
(DOCS / "assets" / "vendor").mkdir(exist_ok=True)
(DOCS / "assets" / "vendor" / "plotly.min.js").write_text(get_plotlyjs(), encoding="utf-8")
print("Bundled", len(figs), "figures for the website")

Bundled 10 figures for the website
